In [21]:
import csv
import json

class Grafo:
    def __init__(self, nodos, matriz=None, estructura=None):
        self.nodos = nodos
        self.matriz = matriz # Matriz de adyacencia para CSV
        self.estructura = estructura # Lista de adyacencia para JSON

    @classmethod
    def desde_csv(cls, filename):

        with open(filename, newline='') as f:
            reader = csv.reader(f)
            matriz = [list(map(int, row)) for row in reader]
        # Los nodos son identificados por su posición en la matriz
        nodos = [str(i) for i in range(len(matriz))]
        return cls(nodos, matriz=matriz)

    @classmethod
    def desde_json(cls, filename):
        with open(filename) as f:
            data = json.load(f)

        clave_nodos = 'nodos' # Clave para los nodos
        clave_aristas = 'aristas' # Clave para las aristas

        if clave_nodos in data and clave_aristas in data:
            nodos = data[clave_nodos]
            aristas = data[clave_aristas]
            estructura_grafo = {'V': nodos, 'E': aristas}

            return cls(nodos, estructura=estructura_grafo)
        else:
            print(f"Error: El archivo JSON '{filename}' no contiene las claves esperadas '{clave_nodos}' o '{clave_aristas}'.")
            return None

    def _adyacentes_derecha(self, nodo):
        """Encuentra los vecinos derechos de un nodo."""
        if self.matriz is not None:
            try:
                idx = self.nodos.index(nodo)
                return [self.nodos[j] for j in range(len(self.matriz[idx])) if self.matriz[idx][j] == 1]
            except ValueError:
                print(f"Error: El nodo '{nodo}' no se encontró en el grafo (matriz).")
                return []
        # Adaptado para usar la estructura interna con 'E' si existe
        elif self.estructura is not None and 'E' in self.estructura:
            return self.estructura['E'].get(nodo, [])
        else:
            return []

    def _adyacentes_izquierda(self, nodo):
        if self.matriz is not None:
            try:
                idx = self.nodos.index(nodo)
                return [self.nodos[i] for i in range(len(self.matriz)) if self.matriz[i][idx] == 1]
            except ValueError:
                print(f"Error: El nodo '{nodo}' no se encontró en el grafo (matriz).")
                return []
        # Adaptado para usar la estructura interna con 'E' si existe
        elif self.estructura is not None and 'E' in self.estructura:
            # Encuentra nodos cuyas listas de adyacencia contengan 'nodo'
            return [u for u, vecinos in self.estructura['E'].items() if nodo in vecinos]
        else:
             return []


    def minimales(self):
        """Encuentra los nodos minimales (sin aristas entrantes)."""
        return [n for n in self.nodos if len(self._adyacentes_izquierda(n)) == 0]


    def maximales(self):
        """Encuentra los nodos maximales (sin aristas salientes)."""
        return [n for n in self.nodos if len(self._adyacentes_derecha(n)) == 0]

    def vecindad_derecha(self, nodo):
        """Retorna la vecindad derecha de un nodo."""
        return self._adyacentes_derecha(nodo)

    def vecindad_izquierda(self, nodo):
        """Retorna la vecindad izquierda de un nodo."""
        return self._adyacentes_izquierda(nodo)

# Ejemplo
print("--- Grafo desde CSV (/content/01.csv) ---")
g_csv = Grafo.desde_csv("/content/01.csv")
if g_csv:
    print("Nodos:", g_csv.nodos)
    print("Matriz de adyacencia:")
    for row in g_csv.matriz:
        print(row)
    print("Minimales:", g_csv.minimales())
    print("Maximales:", g_csv.maximales())
    # Ejemplo de vecindad para un nodo (ej. nodo '0')
    print("Vecindad derecha de '0':", g_csv.vecindad_derecha('0'))
    print("Vecindad izquierda de '2':", g_csv.vecindad_izquierda('2'))

print("\n--- Grafo desde JSON (/content/01.json) ---")
g_json = Grafo.desde_json("/content/01.json")
if g_json:
    print("Nodos:", g_json.nodos)
    # Mostramos la estructura interna si está presente
    if g_json.estructura:
      print("Estructura (Lista de adyacencia interna con 'V' y 'E'):", g_json.estructura)
    print("Minimales:", g_json.minimales())
    print("Maximales:", g_json.maximales())
    if g_json.nodos and g_json.estructura:
         if 'a' in g_json.nodos:
             print("Vecindad derecha de 'a':", g_json.vecindad_derecha('a'))
         else:
             print("Nota: No se pudo probar la vecindad derecha de 'a' ya que 'a' no está en los nodos cargados del JSON.")

         if 'c' in g_json.nodos:
              print("Vecindad izquierda de 'c':", g_json.vecindad_izquierda('c'))
         else:
             print("Nota: No se pudo probar la vecindad izquierda de 'c' ya que 'c' no está en los nodos cargados del JSON.")

--- Grafo desde CSV (/content/01.csv) ---
Nodos: ['0', '1', '2', '3', '4', '5']
Matriz de adyacencia:
[1, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0]
[0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 1, 0]
[0, 0, 0, 0, 0, 1]
Minimales: []
Maximales: []
Vecindad derecha de '0': ['0']
Vecindad izquierda de '2': ['2']

--- Grafo desde JSON (/content/01.json) ---
Error: El archivo JSON '/content/01.json' no contiene las claves esperadas 'nodos' o 'aristas'.
